# Optimizers in an Ill-Conditioned Valley

AI & ML Foundations: Optimizers · *Mathematics for AI and Machine Learning* by Fuheng Wu

Gradient descent, momentum and Adam on a narrow valley; rotate it and Adam's per-axis scaling stops helping (Chapters 1, 11, 20).

Companion code for the [interactive demo](https://math4ai.org/companion/ml/optimizers-valley/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

# The demo's defaults: f(x) = x^T H x / 2 with H = R diag(1, kappa) R^T, start R (-8, 4), 100 steps
# (the start turns with the valley, so only the valley's orientation changes)
kappa, steps = 25.0, 100


def make_problem(theta_deg):
    t = math.radians(theta_deg)
    R = torch.tensor([[math.cos(t), -math.sin(t)], [math.sin(t), math.cos(t)]], dtype=torch.float64)
    H = R @ torch.diag(torch.tensor([1.0, kappa], dtype=torch.float64)) @ R.T
    return H, R @ torch.tensor([-8.0, 4.0], dtype=torch.float64)


def run(opt_name, theta):
    H, start = make_problem(theta)
    x = start.clone().requires_grad_(True)
    lr = 1 / kappa
    opt = {"GD": lambda p: torch.optim.SGD(p, lr=lr),
           "Momentum": lambda p: torch.optim.SGD(p, lr=lr, momentum=0.9),
           "Adam": lambda p: torch.optim.Adam(p, lr=0.5)}[opt_name]([x])
    for _ in range(steps):
        opt.zero_grad()
        (0.5 * x @ H @ x).backward()
        opt.step()
    return (0.5 * x @ H @ x).item()


# ── The same valley at three orientations ───────────────────────────────────
print(f"loss after {steps} steps, kappa = {kappa:.0f} (start loss {0.5 * torch.tensor([-8.0, 4.0]) @ torch.tensor([[1.0, 0], [0, kappa]]) @ torch.tensor([-8.0, 4.0]):.0f})")
print("  valley       GD          Momentum    Adam")
for theta in (0, 20, 45):
    print(f"  rotated {theta:2d}°  " + "  ".join(f"{run(o, theta):.3e}" for o in ("GD", "Momentum", "Adam")))
# GD and momentum give the same numbers at every angle: they are rotation invariant.

# GD's slow direction shrinks by (1 - 1/kappa) per step; momentum ~ (1 - 1/sqrt(kappa)) with tuned beta
print(f"\nGD contraction per step along the flat direction: 1 - 1/kappa = {1 - 1 / kappa:.3f}")
print(f"best heavy-ball rate: (sqrt(kappa) - 1)/(sqrt(kappa) + 1) = {(math.sqrt(kappa) - 1) / (math.sqrt(kappa) + 1):.3f}")
# Adam divides each coordinate's step by its own gradient size: ideal when the valley is
# aligned with the axes (a diagonal metric), not when it is rotated (Chapter 1's table).

loss after 100 steps, kappa = 25 (start loss 232)
  valley       GD          Momentum    Adam


  rotated  0°  9.107e-03  7.264e-04  1.978e-03
  rotated 20°  9.107e-03  7.264e-04  4.758e-03
  rotated 45°  9.107e-03  7.264e-04  1.801e-02

GD contraction per step along the flat direction: 1 - 1/kappa = 0.960
best heavy-ball rate: (sqrt(kappa) - 1)/(sqrt(kappa) + 1) = 0.667
